# 09_phase3_PAGA_trajectory.ipynb
Phase 3 stretch — PAGA trajectory inference

**Scope:** GSE114725 T cell sub-clusters (CD4 Naive/Resting, CD4 Activated, Regulatory T cells, NK/Cytotoxic T cells) — chosen because trajectory inference is most meaningful within a biologically continuous population, and this sub-cluster set was already validated in Phase 2.

**What PAGA does:** builds a graph of connectivity between clusters (not just nearest-neighbour distances within a cluster) — showing which cluster transitions are well-supported by intermediate cells vs which clusters are transcriptionally disconnected. This is a coarse, cluster-level trajectory estimate, not a full pseudotime ordering of individual cells (that would be a natural follow-up if this shows a clear structure worth pursuing further).

In [24]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_paga"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_paga"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


# ----------------------------
# NOTE: this PAGA analysis uses the original, pooled T-cell sub-clustering
# (GSE114725_tcells_subclustered.h5ad), later found to be methodologically
# flawed (Section 2.6.1). The corrected CD8-lineage PAGA analysis, using
# the properly-rebuilt sub-clustering, lives in 21_phase4_finelabels_PAGA.ipynb
# and is the version reported in this thesis. This notebook's output is
# retained for its own record but should not be cited as a validated result.
# ----------------------------

In [25]:
# ----------------------------
# Cell 2 — Load T cell sub-clustered data
# ----------------------------
adata_tcells = sc.read_h5ad(PROCESSED_DIR / "GSE114725_tcells_subclustered.h5ad")
print(f"Loaded: {adata_tcells.n_obs} cells, {adata_tcells.n_vars} genes")
print(adata_tcells.obs["tcell_subtype"].value_counts())

Loaded: 33003 cells, 2000 genes
tcell_subtype
CD4 Naive/Resting T cells           10807
CD4 Activated T cells                8398
Activated CD8 T cells                6256
NK/Cytotoxic T cells                 5542
Regulatory T cells (Tregs, CD4+)     2000
Name: count, dtype: int64


In [26]:
# ----------------------------
# Cell 3 — Run PAGA
# Uses the existing neighbour graph from sub-clustering (already computed
# on X_pca_harmony, seeded, single-threaded — same reproducibility
# standard as the rest of the pipeline). PAGA itself is deterministic
# given a fixed neighbour graph, no additional seeding needed.
# ----------------------------
adata_tcells.obs["tcell_subtype"] = adata_tcells.obs["tcell_subtype"].astype(str).astype("category")

sc.tl.paga(adata_tcells, groups="tcell_subtype")

print("PAGA connectivity matrix (edge weight = confidence of connection):")
paga_connectivities = pd.DataFrame(
    adata_tcells.uns["paga"]["connectivities"].toarray(),
    index=adata_tcells.obs["tcell_subtype"].cat.categories,
    columns=adata_tcells.obs["tcell_subtype"].cat.categories,
)
print(paga_connectivities.round(3))
paga_connectivities.to_csv(RESULTS_DIR / "GSE114725_tcell_paga_connectivities.csv")

PAGA connectivity matrix (edge weight = confidence of connection):
                                  Activated CD8 T cells  \
Activated CD8 T cells                             0.000   
CD4 Activated T cells                             0.252   
CD4 Naive/Resting T cells                         0.030   
NK/Cytotoxic T cells                              0.117   
Regulatory T cells (Tregs, CD4+)                  0.064   

                                  CD4 Activated T cells  \
Activated CD8 T cells                             0.252   
CD4 Activated T cells                             0.000   
CD4 Naive/Resting T cells                         0.207   
NK/Cytotoxic T cells                              0.037   
Regulatory T cells (Tregs, CD4+)                  0.231   

                                  CD4 Naive/Resting T cells  \
Activated CD8 T cells                                 0.030   
CD4 Activated T cells                                 0.207   
CD4 Naive/Resting T cells         

In [27]:
# ----------------------------
# Cell 4 — PAGA graph visualisation
# Line thickness = connectivity strength between cluster pairs.
# Thick lines = well-supported transitions (many intermediate cells);
# thin/absent lines = transcriptionally distinct, no clear intermediate
# population connecting them.
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.paga(adata_tcells, color="tcell_subtype", threshold=0.05,
           node_size_scale=2, edge_width_scale=1.5,
           fontsize=10, frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_tcell_paga_graph.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("PAGA graph saved")

PAGA graph saved


In [28]:
# ----------------------------
# Cell 5 — PAGA-initialised UMAP
# Standard practice: use PAGA's coarse structure to initialise UMAP
# layout, producing a more trajectory-faithful embedding than a
# default random initialisation.
# ----------------------------
sc.tl.umap(adata_tcells, init_pos="paga", random_state=42)

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.umap(adata_tcells, color="tcell_subtype",
           title="GSE114725 T cells — PAGA-initialised UMAP",
           legend_loc="right margin", legend_fontsize=9,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_tcell_paga_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

adata_tcells.write(PROCESSED_DIR / "GSE114725_tcells_paga.h5ad", compression="gzip")
print("PAGA-initialised UMAP saved, object saved with PAGA results embedded")

PAGA-initialised UMAP saved, object saved with PAGA results embedded


In [29]:
# ----------------------------
# Cell 6 — Interpretation summary
# ----------------------------
print("=== PAGA connectivity summary ===")
print("Highest-connectivity pairs (excluding self-connections):\n")

conn = paga_connectivities.copy()
np.fill_diagonal(conn.values, 0)
pairs = []
for i, row in enumerate(conn.index):
    for j, col in enumerate(conn.columns):
        if j > i:
            pairs.append((row, col, conn.iloc[i, j]))
pairs_df = pd.DataFrame(pairs, columns=["Cluster A", "Cluster B", "Connectivity"])
pairs_df = pairs_df.sort_values("Connectivity", ascending=False)
print(pairs_df.to_string(index=False))
pairs_df.to_csv(RESULTS_DIR / "GSE114725_tcell_paga_pairwise_connectivity.csv", index=False)

print("\n>>> Interpretation guide: high connectivity between two clusters suggests")
print(">>> a plausible transition/continuum between them (e.g. Naive -> Activated")
print(">>> would be biologically expected to connect). Low/zero connectivity between")
print(">>> two clusters suggests they are transcriptionally distinct end-states,")
print(">>> not part of the same differentiation path.")

=== PAGA connectivity summary ===
Highest-connectivity pairs (excluding self-connections):

                Cluster A                        Cluster B  Connectivity
    Activated CD8 T cells            CD4 Activated T cells      0.251694
    CD4 Activated T cells Regulatory T cells (Tregs, CD4+)      0.231057
CD4 Naive/Resting T cells Regulatory T cells (Tregs, CD4+)      0.212093
    CD4 Activated T cells        CD4 Naive/Resting T cells      0.206564
    Activated CD8 T cells             NK/Cytotoxic T cells      0.117407
    Activated CD8 T cells Regulatory T cells (Tregs, CD4+)      0.063633
    CD4 Activated T cells             NK/Cytotoxic T cells      0.036717
    Activated CD8 T cells        CD4 Naive/Resting T cells      0.030142
     NK/Cytotoxic T cells Regulatory T cells (Tregs, CD4+)      0.007816
CD4 Naive/Resting T cells             NK/Cytotoxic T cells      0.003306

>>> Interpretation guide: high connectivity between two clusters suggests
>>> a plausible transition/con

## Macrophage trajectory (GSE114725)

Monocyte-to-macrophage differentiation is a well-established biological continuum, making this sub-clustering set arguably a stronger PAGA candidate than the T cell set above. The contamination artefact cluster is excluded — it isn't real macrophage biology and shouldn't be treated as a point on any trajectory.

In [30]:
# ----------------------------
# Cell 7 — Load macrophage sub-clustered data, exclude contamination cluster
# ----------------------------
adata_mac = sc.read_h5ad(PROCESSED_DIR / "GSE114725_macrophages_subclustered.h5ad")
print(f"Loaded: {adata_mac.n_obs} cells")

adata_mac = adata_mac[
    adata_mac.obs["mac_subtype"] != "Unassigned (n=91, stromal/RBC contamination artefact)"
].copy()
print(f"After excluding contamination artefact: {adata_mac.n_obs} cells")
print(adata_mac.obs["mac_subtype"].value_counts())

Loaded: 5914 cells
After excluding contamination artefact: 5823 cells
mac_subtype
Monocyte-like macrophages            1265
Complement-high macrophages          1075
LAM-like macrophages                  946
Antigen-presenting macrophages        910
Lipid-laden/Foam-cell macrophages     735
Resting/Resident macrophages          469
Non-classical monocytes (CD16+)       423
Name: count, dtype: int64


In [31]:
# ----------------------------
# Cell 8 — Run PAGA on macrophage sub-clusters
# ----------------------------
adata_mac.obs["mac_subtype"] = adata_mac.obs["mac_subtype"].astype(str).astype("category")

sc.tl.paga(adata_mac, groups="mac_subtype")

paga_connectivities_mac = pd.DataFrame(
    adata_mac.uns["paga"]["connectivities"].toarray(),
    index=adata_mac.obs["mac_subtype"].cat.categories,
    columns=adata_mac.obs["mac_subtype"].cat.categories,
)
print("PAGA connectivity matrix (macrophages):")
print(paga_connectivities_mac.round(3))
paga_connectivities_mac.to_csv(RESULTS_DIR / "GSE114725_macrophage_paga_connectivities.csv")

PAGA connectivity matrix (macrophages):
                                   Antigen-presenting macrophages  \
Antigen-presenting macrophages                              0.000   
Complement-high macrophages                                 0.080   
LAM-like macrophages                                        0.151   
Lipid-laden/Foam-cell macrophages                           0.015   
Monocyte-like macrophages                                   0.155   
Non-classical monocytes (CD16+)                             0.015   
Resting/Resident macrophages                                0.008   

                                   Complement-high macrophages  \
Antigen-presenting macrophages                           0.080   
Complement-high macrophages                              0.000   
LAM-like macrophages                                     0.234   
Lipid-laden/Foam-cell macrophages                        0.272   
Monocyte-like macrophages                                0.086   
Non-classic

In [32]:
# ----------------------------
# Cell 9 — Macrophage PAGA graph + PAGA-initialised UMAP
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.paga(adata_mac, color="mac_subtype", threshold=0.05,
           node_size_scale=2, edge_width_scale=1.5,
           fontsize=9, frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_macrophage_paga_graph.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

sc.tl.umap(adata_mac, init_pos="paga", random_state=42)
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.umap(adata_mac, color="mac_subtype",
           title="GSE114725 Macrophages — PAGA-initialised UMAP",
           legend_loc="right margin", legend_fontsize=9,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_macrophage_paga_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

adata_mac.write(PROCESSED_DIR / "GSE114725_macrophages_paga.h5ad", compression="gzip")
print("Macrophage PAGA graph, UMAP, and object saved")

Macrophage PAGA graph, UMAP, and object saved


In [33]:
# ----------------------------
# Cell 10 — Macrophage PAGA interpretation summary
# ----------------------------
conn_mac = paga_connectivities_mac.copy()
np.fill_diagonal(conn_mac.values, 0)
pairs_mac = []
for i, row in enumerate(conn_mac.index):
    for j, col in enumerate(conn_mac.columns):
        if j > i:
            pairs_mac.append((row, col, conn_mac.iloc[i, j]))
pairs_mac_df = pd.DataFrame(pairs_mac, columns=["Cluster A", "Cluster B", "Connectivity"])
pairs_mac_df = pairs_mac_df.sort_values("Connectivity", ascending=False)
print("=== Macrophage PAGA connectivity, ranked ===")
print(pairs_mac_df.to_string(index=False))
pairs_mac_df.to_csv(RESULTS_DIR / "GSE114725_macrophage_paga_pairwise_connectivity.csv", index=False)

print("\n>>> Look for: does Monocyte-like / Non-classical monocytes (CD16+) show high")
print(">>> connectivity to intermediate states, consistent with a genuine")
print(">>> monocyte-to-macrophage differentiation trajectory? Or do polarized end-states")
print(">>> (LAM-like, Lipid-laden) sit as separate branches rather than one linear path?")

=== Macrophage PAGA connectivity, ranked ===
                        Cluster A                         Cluster B  Connectivity
             LAM-like macrophages Lipid-laden/Foam-cell macrophages      0.306834
      Complement-high macrophages Lipid-laden/Foam-cell macrophages      0.272129
      Complement-high macrophages              LAM-like macrophages      0.233504
      Complement-high macrophages      Resting/Resident macrophages      0.224409
             LAM-like macrophages      Resting/Resident macrophages      0.172819
   Antigen-presenting macrophages         Monocyte-like macrophages      0.155207
        Monocyte-like macrophages   Non-classical monocytes (CD16+)      0.152021
Lipid-laden/Foam-cell macrophages         Monocyte-like macrophages      0.151678
   Antigen-presenting macrophages              LAM-like macrophages      0.151201
      Complement-high macrophages         Monocyte-like macrophages      0.085655
   Antigen-presenting macrophages       Complement-hi

## Main clustering trajectory (GSE114725, all 9 cell types)

**Caveat, worth reading before the results below:** PAGA computes connectivity between *any* clusters given to it, including pairs with no plausible real developmental relationship (e.g. B cells and Mast cells). A connection appearing here doesn't automatically mean a real trajectory — it means the two populations share more transcriptional neighbours than expected by chance, which could reflect a real biological link (e.g. Monocytes/DC connecting to Macrophages would be expected) or just general similarity between two related but distinct immune lineages. The contamination artefact cluster is excluded, same as the macrophage analysis above.

In [34]:
# ----------------------------
# Cell 11 — Load full annotated GSE114725, exclude contamination cluster
# ----------------------------
adata_main = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
print(f"Loaded: {adata_main.n_obs} cells")

adata_main = adata_main[
    adata_main.obs["cell_type"] != "Mixed/stromal-contaminated (CD8+fibroblast signal)"
].copy()
print(f"After excluding contamination cluster: {adata_main.n_obs} cells")
print(adata_main.obs["cell_type"].value_counts())

Loaded: 44662 cells
After excluding contamination cluster: 43327 cells
cell_type
T cells                 20213
CD8/Effector T cells     7861
Macrophages              5914
NK/Cytotoxic T cells     4929
B cells                  3369
Mast cells                494
Monocytes/DC              359
pDC                       188
Name: count, dtype: int64


In [35]:
# ----------------------------
# Cell 12 — Run PAGA on main clustering
# ----------------------------
sc.pp.neighbors(adata_main, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)
adata_main.obs["cell_type"] = adata_main.obs["cell_type"].astype(str).astype("category")

sc.tl.paga(adata_main, groups="cell_type")

paga_connectivities_main = pd.DataFrame(
    adata_main.uns["paga"]["connectivities"].toarray(),
    index=adata_main.obs["cell_type"].cat.categories,
    columns=adata_main.obs["cell_type"].cat.categories,
)
print("PAGA connectivity matrix (main clustering):")
print(paga_connectivities_main.round(3))
paga_connectivities_main.to_csv(RESULTS_DIR / "GSE114725_main_paga_connectivities.csv")

PAGA connectivity matrix (main clustering):
                      B cells  CD8/Effector T cells  Macrophages  Mast cells  \
B cells                 0.000                 0.002        0.009       0.000   
CD8/Effector T cells    0.002                 0.000        0.004       0.008   
Macrophages             0.009                 0.004        0.000       0.005   
Mast cells              0.000                 0.008        0.005       0.000   
Monocytes/DC            0.000                 0.021        0.057       0.008   
NK/Cytotoxic T cells    0.001                 0.166        0.000       0.002   
T cells                 0.005                 0.128        0.004       0.014   
pDC                     0.137                 0.003        0.104       0.000   

                      Monocytes/DC  NK/Cytotoxic T cells  T cells    pDC  
B cells                      0.000                 0.001    0.005  0.137  
CD8/Effector T cells         0.021                 0.166    0.128  0.003  
Macrophage

In [36]:
# ----------------------------
# Cell 13 — Main clustering PAGA graph + PAGA-initialised UMAP
# ----------------------------
fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.paga(adata_main, color="cell_type", threshold=0.05,
           node_size_scale=2, edge_width_scale=1.5,
           fontsize=9, frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_main_paga_graph.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

sc.tl.umap(adata_main, init_pos="paga", random_state=42)
fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.umap(adata_main, color="cell_type",
           title="GSE114725 Main clustering — PAGA-initialised UMAP",
           legend_loc="right margin", legend_fontsize=8,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_main_paga_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

adata_main.write(PROCESSED_DIR / "GSE114725_main_paga.h5ad", compression="gzip")
print("Main clustering PAGA graph, UMAP, and object saved")

Main clustering PAGA graph, UMAP, and object saved


In [37]:
# ----------------------------
# Cell 14 — Main clustering PAGA interpretation summary
# ----------------------------
conn_main = paga_connectivities_main.copy()
np.fill_diagonal(conn_main.values, 0)
pairs_main = []
for i, row in enumerate(conn_main.index):
    for j, col in enumerate(conn_main.columns):
        if j > i:
            pairs_main.append((row, col, conn_main.iloc[i, j]))
pairs_main_df = pd.DataFrame(pairs_main, columns=["Cluster A", "Cluster B", "Connectivity"])
pairs_main_df = pairs_main_df.sort_values("Connectivity", ascending=False)
print("=== Main clustering PAGA connectivity, ranked (top 15) ===")
print(pairs_main_df.head(15).to_string(index=False))
pairs_main_df.to_csv(RESULTS_DIR / "GSE114725_main_paga_pairwise_connectivity.csv", index=False)

print("\n>>> Reminder: read this against known biology, not at face value.")
print(">>> Monocytes/DC <-> Macrophages connecting would be expected (both myeloid,")
print(">>> plausible differentiation relationship). T cells <-> CD8/Effector T cells")
print(">>> connecting would be expected (shared lineage, activation continuum).")
print(">>> A high connection between two otherwise unrelated lineages (e.g. B cells")
print(">>> <-> Mast cells) would be more likely a shared technical/transcriptional")
print(">>> similarity than a real trajectory, and should be reported cautiously if")
print(">>> it appears.")

=== Main clustering PAGA connectivity, ranked (top 15) ===
           Cluster A            Cluster B  Connectivity
CD8/Effector T cells NK/Cytotoxic T cells      0.165665
             B cells                  pDC      0.136811
CD8/Effector T cells              T cells      0.128386
         Macrophages                  pDC      0.103509
         Macrophages         Monocytes/DC      0.057394
CD8/Effector T cells         Monocytes/DC      0.020630
        Monocytes/DC NK/Cytotoxic T cells      0.015303
          Mast cells              T cells      0.013966
        Monocytes/DC              T cells      0.010262
             B cells          Macrophages      0.009038
             T cells                  pDC      0.008551
CD8/Effector T cells           Mast cells      0.007670
          Mast cells         Monocytes/DC      0.007634
NK/Cytotoxic T cells              T cells      0.007447
             B cells              T cells      0.004851

>>> Reminder: read this against known biolog